## Embedding
Source: https://lena-voita.github.io/nlp_course/word_embeddings.html

In [2]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from collections import Counter, defaultdict

# Sample corpus
corpus = ["I like machine learning", "machine learning is fun", "I enjoy deep learning"]

# Step 1: Tokenize and build vocabulary
vocab = set(word for sentence in corpus for word in sentence.split())
vocab_to_index = {word: i for i, word in enumerate(vocab)}
index_to_vocab = {i: word for word, i in vocab_to_index.items()}

# Step 2: Define window size and initialize co-occurrence matrix
window_size = 2
co_occurrence_matrix = np.zeros((len(vocab), len(vocab)), dtype=np.int32)

# Step 3: Populate the co-occurrence matrix
for sentence in corpus:
    words = sentence.split()
    for i, word in enumerate(words):
        word_idx = vocab_to_index[word]
        start = max(0, i - window_size)
        end = min(len(words), i + window_size + 1)
        for j in range(start, end):
            if i != j:  # Exclude the word itself
                context_word = words[j]
                context_idx = vocab_to_index[context_word]
                co_occurrence_matrix[word_idx][context_idx] += 1

# Step 4: Apply SVD to reduce dimensionality
svd = TruncatedSVD(n_components=5)  # Adjust based on vocabulary size
embeddings = svd.fit_transform(co_occurrence_matrix)

# Now, `embeddings` holds the low-dimensional representation of each word
print("Embedding for each word:")
for word, idx in vocab_to_index.items():
    print(f"{word}: {embeddings[idx]}")


Embedding for each word:
machine: [ 2.02072625e+00  1.49847882e+00  3.00053269e-01  6.64114502e-01
 -4.36720682e-16]
is: [ 1.22536528e+00 -9.54698539e-02  7.44619530e-01 -8.66458006e-01
 -4.29757022e-16]
fun: [8.37809037e-01 6.54439849e-01 6.30786120e-01 4.76018892e-01
 2.59502152e-16]
deep: [ 1.13292255  0.7621242  -0.70706037 -0.19762324 -0.70710678]
learning: [ 2.30517928e+00 -1.86667978e+00  2.87282290e-01  1.54955696e-01
  5.55111512e-16]
like: [ 1.34360109e+00  5.16526307e-01 -9.14268934e-02 -7.74154670e-01
 -2.99269115e-18]
I: [ 1.33605716e+00 -1.18045459e+00 -7.36900641e-01  3.38053702e-01
  2.22044605e-16]
enjoy: [ 1.13292255  0.7621242  -0.70706037 -0.19762324  0.70710678]


In [5]:
co_occurrence_matrix 

array([[0, 1, 0, 0, 2, 1, 1, 0],
       [1, 0, 1, 0, 1, 0, 0, 0],
       [0, 1, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 1, 1],
       [2, 1, 1, 1, 0, 1, 0, 1],
       [1, 0, 0, 0, 1, 0, 1, 0],
       [1, 0, 0, 1, 0, 1, 0, 1],
       [0, 0, 0, 1, 1, 0, 1, 0]], dtype=int32)

In [13]:
a = np.array([[3],[1],[2]])
b = np.array([[5 ,3,10]])


In [14]:
a.shape

(3, 1)

In [15]:
b.shape

(1, 3)

In [18]:
a @ b

array([[15,  9, 30],
       [ 5,  3, 10],
       [10,  6, 20]])

In [20]:
np.linalg.matrix_rank(a @ b)

np.int64(1)

In [21]:
## lets get the singular value decomposition of the matrix a @ b
u, sigma, v = np.linalg.svd(a @ b)

In [23]:
import torch
?torch.linalg.svd

Docstring:
linalg.svd(A, full_matrices=True, *, driver=None, out=None) -> (Tensor, Tensor, Tensor)

Computes the singular value decomposition (SVD) of a matrix.

Letting :math:`\mathbb{K}` be :math:`\mathbb{R}` or :math:`\mathbb{C}`,
the **full SVD** of a matrix
:math:`A \in \mathbb{K}^{m \times n}`, if `k = min(m,n)`, is defined as

.. math::

    A = U \operatorname{diag}(S) V^{\text{H}}
    \mathrlap{\qquad U \in \mathbb{K}^{m \times m}, S \in \mathbb{R}^k, V \in \mathbb{K}^{n \times n}}

where :math:`\operatorname{diag}(S) \in \mathbb{K}^{m \times n}`,
:math:`V^{\text{H}}` is the conjugate transpose when :math:`V` is complex, and the transpose when :math:`V` is real-valued.
The matrices  :math:`U`, :math:`V` (and thus :math:`V^{\text{H}}`) are orthogonal in the real case, and unitary in the complex case.

When `m > n` (resp. `m < n`) we can drop the last `m - n` (resp. `n - m`) columns of `U` (resp. `V`) to form the **reduced SVD**:

.. math::

    A = U \operatorname{diag}(S) V^{\

In [32]:
embedding_dim = 100  # Size of word embeddings
context_size = 2  # Number of words on either side of target word
epochs = 10  # Number of epochs
learning_rate = 0.001


In [67]:
corpus = [
    "we are learning word embeddings",
    "word embeddings are useful for natural language processing",
    "we will learn how to use pytorch"
]
# Tokenize and prepare vocabulary
words = " ".join(corpus).split()
vocab = set(words)
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for i, word in enumerate(vocab)}

In [69]:
#ix_to_word

In [38]:
def make_skipgram_data(corpus, context_size):
    data = []
    for sentence in corpus:
        tokens = sentence.split()
        for i, target in enumerate(tokens):
            context = tokens[max(0, i - context_size):i] + tokens[i + 1:i + 1 + context_size]
            print(f"Context: {context}, Target: {target}")
            target_idx = word_to_ix[target]
            context_idx = [word_to_ix[word] for word in context]
            for context_word in context_idx:
                data.append((target_idx, context_word))
    return data

training_data = make_skipgram_data(corpus, context_size)


Context: ['are', 'learning'], Target: we
Context: ['we', 'learning', 'word'], Target: are
Context: ['we', 'are', 'word', 'embeddings'], Target: learning
Context: ['are', 'learning', 'embeddings'], Target: word
Context: ['learning', 'word'], Target: embeddings
Context: ['embeddings', 'are'], Target: word
Context: ['word', 'are', 'useful'], Target: embeddings
Context: ['word', 'embeddings', 'useful', 'for'], Target: are
Context: ['embeddings', 'are', 'for', 'natural'], Target: useful
Context: ['are', 'useful', 'natural', 'language'], Target: for
Context: ['useful', 'for', 'language', 'processing'], Target: natural
Context: ['for', 'natural', 'processing'], Target: language
Context: ['natural', 'language'], Target: processing
Context: ['will', 'learn'], Target: we
Context: ['we', 'learn', 'how'], Target: will
Context: ['we', 'will', 'how', 'to'], Target: learn
Context: ['will', 'learn', 'to', 'use'], Target: how
Context: ['learn', 'how', 'use', 'pytorch'], Target: to
Context: ['how', 'to'

In [70]:
[ {ix_to_word[i[1]]: ix_to_word[i[0]]} for i in training_data]

[{'are': 'we'},
 {'learning': 'we'},
 {'we': 'are'},
 {'learning': 'are'},
 {'word': 'are'},
 {'we': 'learning'},
 {'are': 'learning'},
 {'word': 'learning'},
 {'embeddings': 'learning'},
 {'are': 'word'},
 {'learning': 'word'},
 {'embeddings': 'word'},
 {'learning': 'embeddings'},
 {'word': 'embeddings'},
 {'embeddings': 'word'},
 {'are': 'word'},
 {'word': 'embeddings'},
 {'are': 'embeddings'},
 {'useful': 'embeddings'},
 {'word': 'are'},
 {'embeddings': 'are'},
 {'useful': 'are'},
 {'for': 'are'},
 {'embeddings': 'useful'},
 {'are': 'useful'},
 {'for': 'useful'},
 {'natural': 'useful'},
 {'are': 'for'},
 {'useful': 'for'},
 {'natural': 'for'},
 {'language': 'for'},
 {'useful': 'natural'},
 {'for': 'natural'},
 {'language': 'natural'},
 {'processing': 'natural'},
 {'for': 'language'},
 {'natural': 'language'},
 {'processing': 'language'},
 {'natural': 'processing'},
 {'language': 'processing'},
 {'will': 'we'},
 {'learn': 'we'},
 {'we': 'will'},
 {'learn': 'will'},
 {'how': 'will'},


In [66]:
#index_to_vocab

In [43]:
import torch.nn as nn
embeddings = nn.Embedding(len(vocab), embedding_dim)

In [48]:
for k,v in embeddings.named_parameters():
    print(k)

weight


In [ ]:
embeddings.state_dict()

OrderedDict([('weight',
              tensor([[-1.2279, -0.3868,  0.4180,  ..., -1.3406, -0.1450, -0.7047],
                      [-0.0413,  1.2201, -0.4092,  ...,  1.1445,  1.4282, -0.4637],
                      [ 1.4307, -0.0566,  0.2196,  ..., -0.6871, -1.3889, -0.0633],
                      ...,
                      [ 0.4674,  0.6167, -0.6527,  ...,  1.2933,  1.2161, -0.0214],
                      [-0.6909, -1.0753,  0.3963,  ...,  1.7346,  0.7106,  1.3839],
                      [ 1.9914, -0.6486, -0.6075,  ...,  0.5109, -2.5435, -0.3385]]))])

In [76]:
embeddings.weight

Parameter containing:
tensor([[-1.2279, -0.3868,  0.4180,  ..., -1.3406, -0.1450, -0.7047],
        [-0.0413,  1.2201, -0.4092,  ...,  1.1445,  1.4282, -0.4637],
        [ 1.4307, -0.0566,  0.2196,  ..., -0.6871, -1.3889, -0.0633],
        ...,
        [ 0.4674,  0.6167, -0.6527,  ...,  1.2933,  1.2161, -0.0214],
        [-0.6909, -1.0753,  0.3963,  ...,  1.7346,  0.7106,  1.3839],
        [ 1.9914, -0.6486, -0.6075,  ...,  0.5109, -2.5435, -0.3385]],
       requires_grad=True)

In [83]:
token_tensor = torch.tensor([word_to_ix[word] for word in corpus[1].split()])

In [84]:
embedding_dim = 5  # Example embedding dimension
embedding_layer = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embedding_dim)


In [88]:
embedding_layer(torch.tensor([0]))

tensor([[ 1.4548,  1.0435, -0.1196, -0.7686,  1.4339]],
       grad_fn=<EmbeddingBackward0>)

In [ ]:
## train a word to vec model in pytroch 

import torch
import torch.nn as nn

class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.in_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.out_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        
    def forward(self, target, context):
        emb_target = self.in_embed(target)
        emb_context = self.out_embed(context)
        scores = torch.mm(emb_target, emb_context.T)
        return scores
    
vocab_size = len(vocab)
model = SkipGramModel(vocab_size, embedding_dim)

target = torch.tensor([word_to_ix["word"]], dtype=torch.long)
context = torch.tensor([word_to_ix["embeddings"]], dtype=torch.long)

model(target, context)

import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


for epoch in range(epochs):
    total_loss = 0
    for target, context in training_data:
        target_tensor = torch.tensor([target], dtype=torch.long)
        context_tensor = torch.tensor([context], dtype=torch.long)
        model.zero_grad()
        scores = model(target_tensor, context_tensor)
        loss = loss_fn(scores, context_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss}")

In [2]:

from datasets import load_dataset
imdb_dataset = load_dataset("stanfordnlp/imdb")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
imdb_dataset.keys()

dict_keys(['train', 'test', 'unsupervised'])

In [9]:
from collections import Counter
Counter(imdb_dataset['train']['label'])

Counter({0: 12500, 1: 12500})

In [10]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [17]:
import pprint
START_TOKEN = '<START>'
END_TOKEN = '<END>'
NUM_SAMPLES = 150
import re
def read_corpus():
    """ Read files from the Large Movie Review Dataset.
        Params:
            category (string): category name
        Return:
            list of lists, with words from each of the processed files
    """
    files = imdb_dataset["train"]["text"][:-1]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]

imdb_corpus = read_corpus()
pprint.pprint(imdb_corpus[:3], compact=True, width=100)
print("corpus size: ", len(imdb_corpus[0]))

[['<START>', 'i', 'rented', 'i', 'am', 'curiousyellow', 'from', 'my', 'video', 'store', 'because',
  'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first',
  'released', 'in', '1967', 'i', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized',
  'by', 'us', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', 'therefore',
  'being', 'a', 'fan', 'of', 'films', 'considered', 'controversial', 'i', 'really', 'had', 'to',
  'see', 'this', 'for', 'myselfbr', 'br', 'the', 'plot', 'is', 'centered', 'around', 'a', 'young',
  'swedish', 'drama', 'student', 'named', 'lena', 'who', 'wants', 'to', 'learn', 'everything',
  'she', 'can', 'about', 'life', 'in', 'particular', 'she', 'wants', 'to', 'focus', 'her',
  'attentions', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average',
  'swede', 'thought', 'about', 'certain', 'political', 'issues', 'such', 'as', 'the', 'vietnam',
  'war', 'and', 'race', 'issu

In [18]:
from collections import Counter

def distinct_words(corpus):
    """ Determine a list of distinct words for the corpus.
        Params:
            corpus (list of list of strings): corpus of documents
        Return:
            corpus_words (list of strings): sorted list of distinct words across the corpus
            n_corpus_words (integer): number of distinct words across the corpus
    """
    corpus_words = []
    n_corpus_words = -1
    corpus_word_set = set()
    
    # ------------------
    # Write your implementation here.
    
    
    
    # ------------------

    return corpus_words, n_corpus_words

In [27]:
import torch
a = torch.randint(1,10, size=(10, 15), dtype=torch.float32)
a

tensor([[6., 1., 7., 1., 6., 2., 6., 4., 3., 7., 1., 9., 7., 5., 1.],
        [7., 1., 4., 3., 4., 1., 3., 9., 9., 2., 7., 6., 8., 5., 8.],
        [5., 9., 4., 9., 9., 2., 3., 9., 9., 8., 8., 4., 2., 5., 4.],
        [3., 8., 3., 4., 4., 4., 4., 7., 4., 1., 3., 9., 5., 6., 2.],
        [3., 4., 3., 6., 8., 6., 7., 3., 9., 1., 7., 5., 4., 8., 1.],
        [2., 2., 8., 7., 5., 2., 9., 3., 2., 9., 7., 2., 1., 7., 4.],
        [3., 1., 1., 8., 5., 9., 3., 1., 8., 9., 1., 8., 4., 2., 6.],
        [8., 3., 8., 6., 4., 9., 5., 4., 6., 4., 9., 3., 9., 6., 2.],
        [8., 1., 3., 4., 6., 4., 8., 5., 1., 8., 6., 1., 8., 3., 3.],
        [7., 9., 5., 4., 5., 3., 1., 2., 7., 2., 8., 7., 4., 8., 8.]])

In [39]:
print(torch.mean(a,dim=1, keepdim=True))
torch.mean((a - torch.mean(a,dim=1, keepdim=True)),dim=1)

tensor([[4.4000],
        [5.1333],
        [6.0000],
        [4.4667],
        [5.0000],
        [4.6667],
        [4.6000],
        [5.7333],
        [4.6000],
        [5.3333]])


tensor([-9.5367e-08,  1.2716e-07,  0.0000e+00, -3.1789e-08,  0.0000e+00,
         1.5895e-07,  9.5367e-08,  2.2252e-07,  9.5367e-08, -2.8610e-07])

In [2]:
import tikitoken

ModuleNotFoundError: No module named 'tikitoken'

In [4]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

In [10]:
enc.encode("jack and jill went up the hill")

[23223, 326, 441, 492, 5981, 869, 290, 32306]

In [ ]:
enc.decode([23223, 326, 441, 492])

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3335355712.py, line 1)

In [17]:
enc.encode("manikam")

[2309, 168250]

In [15]:
enc.decode([2309,168250])

'manikam'

In [18]:
enc.encode("kalaiyarasan")

[23466, 1361, 22978, 18476]

In [19]:
enc.encode("prabhukiran")

[638, 25482, 1160, 19019]

In [20]:
enc.encode("sayan")

[82, 13348]

In [22]:
enc.decode([13348])

'ayan'